<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/06_pandas_for_omics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Pandas for omics data

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## Learning objectives
- load and **join** a feature (ASV) table, its **taxonomy** and **sample metadata**;
- inspect, check **dtypes**, and handle **missing values**;
- **select / subset** with `[]`, `.loc`, `.iloc` and boolean masks;
- **`groupby`** environment & treatment; aggregate to **phylum** level;
- compute **relative abundance**, **alpha diversity** and a **correlation**;
- **plot** and **export** results.

Dataset: a synthetic amplicon study of **120 ASVs × 48 samples** across three environments (Soil, Freshwater, Sediment) and two treatments (Control, Amended).

> **How to read this notebook while teaching.** Every code cell is commented: the lines starting with `#` explain what the code does and why it matters biologically, so you can narrate the class straight from the cell even without notes. Blockquotes marked **Instructor note** are extra talking points.

---
| Notebook | Topic |
|---|---|
| 01 | Python essentials & operators |
| 02 | Strings & biological sequences |
| 03 | Data structures |
| 04 | Control flow & functions |
| 05 | Files, scripting & modules |
| 06 | Pandas for omics |

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


## 1. Load the three tables
A real amplicon project is *three* linked tables: the counts, the taxonomy of each feature, and the metadata of each sample.

In [2]:
# pandas is THE library for tables. Convention: import it as 'pd'.
import pandas as pd

# read_csv loads a CSV into a DataFrame (a spreadsheet-like table).
# index_col=... tells pandas which column labels the rows.
asv = pd.read_csv('../data/asv_table.csv', index_col='ASV_id')       # counts: ASVs x samples
taxonomy = pd.read_csv('../data/taxonomy.csv', index_col='ASV_id')   # what each ASV IS
meta = pd.read_csv('../data/sample_metadata.csv', index_col='sample_id')  # about each sample
print('asv     :', asv.shape, '(ASVs x samples)')
print('taxonomy:', taxonomy.shape)
print('meta    :', meta.shape)

asv     : (120, 48) (ASVs x samples)
taxonomy: (120, 6)
meta    : (48, 7)


> **Instructor note.** Stress the three-table structure: this is exactly what QIIME2, phyloseq and DADA2 output. Everything downstream is joining these.

In [3]:
asv.head()      # .head() shows the first 5 rows

,S001,S002,S003,S004,S005,S006,S007,S008,S009,S010,...,S039,S040,S041,S042,S043,S044,S045,S046,S047,S048
ASV_id,,,,,,,,,,,,,,,,,,,,,
ASV_001,17,29.0,17.0,33,25,11.0,8,6.0,93,46.0,...,11.0,53,44.0,57,3,132,111.0,108.0,53,13.0
ASV_002,3,11.0,2.0,3,8,0.0,4,7.0,6,9.0,...,18.0,9,5.0,14,0,11,22.0,0.0,9,8.0
ASV_003,12,34.0,9.0,9,13,6.0,12,16.0,13,7.0,...,10.0,0,7.0,4,0,5,0.0,11.0,5,6.0
ASV_004,3,0.0,3.0,2,6,3.0,10,0.0,11,2.0,...,4.0,0,2.0,3,0,3,3.0,4.0,0,4.0
ASV_005,8,0.0,11.0,6,0,10.0,21,4.0,20,0.0,...,48.0,66,28.0,56,4,34,100.0,66.0,291,126.0


In [4]:
taxonomy.head()

,Domain,Phylum,Class,Order,Family,Genus
ASV_id,,,,,,
ASV_001,Bacteria,Nitrospirota,Nitrospiria,Nitrospirales,Nitrospiraceae,Nitrospira
ASV_002,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Bacteroidaceae,Bacteroides
ASV_003,Bacteria,Actinomycetota,Actinomycetia,Mycobacteriales,Mycobacteriaceae,Rhodococcus
ASV_004,Bacteria,Verrucomicrobiota,Verrucomicrobiae,Verrucomicrobiales,Akkermansiaceae,Akkermansia
ASV_005,Archaea,Euryarchaeota,Methanomicrobia,Methanosarcinales,Methanosarcinaceae,Methanobrevibacter


In [5]:
meta.head()

,environment,treatment,replicate,pH,temperature_C,depth_cm,DOC_mg_L
sample_id,,,,,,,
S001,Soil,Control,1,6.99,13.0,15.7,4.57
S002,Soil,Control,2,5.64,11.5,8.2,4.99
S003,Soil,Control,3,5.91,12.7,9.4,3.40
S004,Soil,Control,4,6.36,14.9,3.3,3.51
S005,Soil,Control,5,6.09,15.7,7.6,0.55


## 2. Inspect & data types
First questions with any table: how big, what types, does it look sane?

In [6]:
print(asv.shape)                     # (rows, columns) = (ASVs, samples)
print('dtypes:')
print(asv.dtypes.value_counts())     # what TYPE is each column? (int? float?)
asv.describe().iloc[:, :4]            # per-column summary stats (first 4 samples)

(120, 48)
dtypes:
int64      32
float64    16
Name: count, dtype: int64


,S001,S002,S003,S004
count,120.00000,119.000000,119.000000,120.000000
mean,16.27500,19.210084,17.764706,20.025000
std,17.73946,22.489875,19.354720,21.787153
min,0.00000,0.000000,0.000000,0.000000
25%,4.00000,4.000000,3.000000,5.750000
50%,11.00000,12.000000,12.000000,12.000000
75%,21.25000,29.000000,26.500000,27.250000
max,110.00000,174.000000,102.000000,131.000000


Columns are `float64` because of **missing values** (a single `NaN` forces the whole column to float). Let's confirm and fix that.

## 3. Missing values
Find the gaps, then decide: for count data an absent measurement is **0**.

In [ ]:
# .isnull() marks every empty (NaN) cell True; .sum().sum() totals them.
print('total NaN cells:', int(asv.isnull().sum().sum()))
print('columns with NaN:', (asv.isnull().sum() > 0).sum())

In [ ]:
# For COUNT data, a missing observation means 'not seen' = 0.
asv = asv.fillna(0).astype(int)      # fill NaNs with 0, then convert back to integers
print('after cleaning, any NaN left?', asv.isnull().any().any())
print('dtypes now:', asv.dtypes.unique())   # all int now

## 4. Selecting & subsetting
Columns are samples; rows are ASVs. Use `.loc` (labels) and `.iloc` (positions).

In [ ]:
asv['S001'].head()                   # [ ] selects a COLUMN (one sample)

In [ ]:
asv.loc['ASV_001'].head()            # .loc selects a ROW BY LABEL (one ASV)

In [ ]:
asv.iloc[0:3, 0:5]                    # .iloc selects BY POSITION: rows 0-2, cols 0-4

**Boolean subsetting** — keep only abundant ASVs (total > 500 reads):

In [ ]:
# Build a True/False mask, then use it to filter rows. This is a core pandas pattern.
totals = asv.sum(axis=1)             # axis=1 = sum ACROSS columns -> per-ASV total
abundant = asv[totals > 500]         # keep only rows where the mask is True
print(f'{abundant.shape[0]} of {asv.shape[0]} ASVs exceed 500 reads')

Subset the **metadata** by criteria — e.g. all acidic soil samples:

In [ ]:
# Combine conditions with & (and) / | (or). Each condition needs ( ).
acidic_soil = meta[(meta['environment'] == 'Soil') & (meta['pH'] < 6.5)]
print(acidic_soil.shape[0], 'acidic soil samples')
acidic_soil.head()

Quick category counts with `value_counts`:

In [ ]:
print(meta['environment'].value_counts())    # how many samples per environment
print(taxonomy['Phylum'].value_counts())      # how many ASVs per phylum

## 5. `groupby` — compare communities
To average by sample category we need samples as **rows**, so we transpose the ASV table and attach the metadata.

In [ ]:
# .T transposes: swap rows and columns so SAMPLES become rows.
samples = asv.T                       # now rows = samples, cols = ASVs
# .join() glues the metadata on, matching by the shared index (sample_id).
samples = samples.join(meta)          # add environment, treatment, pH, ...
samples[['environment', 'treatment', 'pH']].head()

In [ ]:
# groupby('environment') makes one group per environment;
# then we sum the ASV columns to get total reads per environment.
asv_cols = list(asv.index)            # the ASV column names after transpose
samples.groupby('environment')[asv_cols].sum().sum(axis=1)

In [ ]:
# The same verb, a different question: total reads per TREATMENT (Control vs Amended).
# groupby works on ANY category column - just swap the grouping key.
samples.groupby('treatment')[asv_cols].sum().sum(axis=1)

### Aggregate to phylum level
A biologically meaningful view: relative abundance **per phylum per environment**. We join the ASV means to their taxonomy.

In [ ]:
# 1) relative abundance: divide each sample (column) by its own total -> fractions.
rel = asv / asv.sum(axis=0)           # axis=0 = down each column

# 2) mean relative abundance per environment, per ASV:
rel_env = rel.T.join(meta['environment']).groupby('environment').mean().T

# 3) collapse ASVs into their PHYLUM using the taxonomy, and sum within phylum:
rel_env_phylum = rel_env.join(taxonomy['Phylum']).groupby('Phylum').sum()
(rel_env_phylum * 100).round(1)       # show as percentages

Read the table for the habitat specialists you would expect: methanogenic *Euryarchaeota* dominate **Sediment** (~37%), *Acidobacteriota* peak in **Soil** (~25%), and *Cyanobacteriota* are strongly enriched in **Freshwater** (their highest habitat by far). *Bacteroidota* are the single most abundant phylum in Freshwater, but the *Cyanobacteriota* signal is the one we follow up with a pH correlation in section 7.

> **Instructor note.** This is the moment to connect back to ecology: the code recovered known habitat preferences purely from the count table and taxonomy.

## 6. Alpha diversity
Two classic per-sample metrics: **observed richness** (number of ASVs present) and the **Shannon index**.

In [ ]:
import numpy as np

# Richness = how many ASVs are present (count > 0) in each sample.
richness = (asv > 0).sum(axis=0)          # (asv>0) is a True/False table; sum counts Trues

# Shannon index: -sum(p * ln p) over the present taxa in a sample.
def shannon(col):
    p = col[col > 0] / col.sum()          # relative abundances of present taxa
    return -(p * np.log(p)).sum()

# .apply runs our function on every column (sample).
shannon_idx = asv.apply(shannon, axis=0)

# Bundle both metrics into a small table and attach the metadata:
diversity = pd.DataFrame({'richness': richness, 'shannon': shannon_idx})
diversity = diversity.join(meta[['environment', 'treatment']])
diversity.head()

In [ ]:
# Does diversity differ by environment? group and average.
diversity.groupby('environment')[['richness', 'shannon']].mean().round(2)

## 7. Correlation
Does any phylum track an environmental gradient? Correlate *Cyanobacteriota* relative abundance with **pH** across samples.

In [ ]:
# Pick the ASVs that are Cyanobacteriota, sum their relative abundance per sample.
cyano_asvs = taxonomy.index[taxonomy['Phylum'] == 'Cyanobacteriota']
cyano_rel = rel.loc[cyano_asvs].sum(axis=0)     # per-sample cyanobacterial fraction
# .corr() gives the Pearson correlation coefficient (-1..+1).
print('Pearson r (Cyanobacteriota vs pH):',
      round(cyano_rel.corr(meta['pH']), 2))

In [ ]:
# A second correlation, a different biological angle: does community RICHNESS
# track temperature across samples? Reuse the alpha-diversity table from section 6.
# (A negative r would mean fewer taxa in warmer samples.)
r_temp = richness.corr(meta['temperature_C'])
print('Pearson r (richness vs temperature):', round(r_temp, 2))

# You can also correlate two environmental variables directly - here pH vs DOC.
print('Pearson r (pH vs DOC):', round(meta['pH'].corr(meta['DOC_mg_L']), 2))

## 8. Plotting
`pandas` plots via matplotlib. Community composition per environment as a stacked bar chart:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# .plot(kind='bar', stacked=True) draws a stacked bar - one bar per environment.
ax = rel_env_phylum.T.plot(kind='bar', stacked=True, figsize=(9, 5))
ax.set_ylabel('Mean relative abundance')
ax.set_xlabel('Environment')
ax.set_title('Phylum composition across environments')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)  # move legend outside
plt.tight_layout()
plt.show()

In [ ]:
# A boxplot shows the DISTRIBUTION of richness within each environment.
diversity.boxplot(column='richness', by='environment', figsize=(7, 4))
plt.title('Observed richness by environment'); plt.suptitle('')
plt.ylabel('number of ASVs'); plt.tight_layout(); plt.show()

## 9. Export results
Save a table you produced so the next step (or a colleague) can reuse it.

In [ ]:
# to_csv writes a DataFrame back to disk - the inverse of read_csv.
(rel_env_phylum * 100).round(2).to_csv('phylum_by_environment.csv')
diversity.to_csv('alpha_diversity.csv')
print('exported phylum_by_environment.csv and alpha_diversity.csv')

---
## Exercises
**E1.** How many **total reads** were sequenced in sample `S010`? (Sum its column.)

**E2.** Using boolean subsetting on `meta`, list the `sample_id`s of all **Sediment** samples with `temperature_C` below 12.

**E3.** Compute the mean **DOC (`DOC_mg_L`)** per **treatment** with `groupby`. Is the Amended group higher?

**E4.** Which **phylum** has the highest mean relative abundance in **Soil**? (Use `rel_env_phylum`.)

**E5.** *(stretch)* Compute mean **Shannon** diversity per treatment. Does amendment change it?

In [ ]:
# your code here

<details>
<summary><b>Solution: E1-E5</b> (click to expand)</summary>

```python
print('S010 total reads:', int(asv['S010'].sum()))

cold_sed = meta[(meta['environment'] == 'Sediment') & (meta['temperature_C'] < 12)]
print(list(cold_sed.index))

print(samples.groupby('treatment')['DOC_mg_L'].mean().round(2))

print('top soil phylum:', rel_env_phylum['Soil'].idxmax())

print(diversity.groupby('treatment')['shannon'].mean().round(2))
```
</details>

## Reproducibility
Everything today lived in **notebooks**: code, results and explanation together. For the rest of the Summer School:
- keep raw data **read-only**, load with relative paths (`../data/`);
- record the environment: `conda env export > environment.yml`;
- **Restart Kernel, Run All** before sharing; it must run top-to-bottom;
- version notebooks with **git** (you will cover this on 16.07).

### Recap
- three linked tables (counts / taxonomy / metadata): join them.
- clean with `fillna`; select with `[]`, `.loc`, `.iloc`, boolean masks.
- **`groupby`** plus a taxonomy join gives a community comparison; `.apply` for diversity.
- normalise to relative abundance, correlate, plot, export.

This completes the Python refresher. You can now read sequences, write functions, and wrangle omics tables in Python.